# FinTech Credit & Loan Review — Relational Model + Quality Reports

**Before running:** upload 3 files to this Colab session (the folder icon on the left sidebar):
- `applicant.csv` — use the well-formed version (the one where the last row, APP-00060, has all 12 fields, most of them empty, not just 3 fields)
- `loan_product.csv`
- `credit_pull.csv`

This notebook is intentionally *not* wrapped in functions — every step is its own cell so you can run one at a time and inspect the output before moving on.

In [1]:
import duckdb

con = duckdb.connect("fintech.duckdb")

## Step 1: Load raw files into staging tables

`applicant.csv` loads with no `ignore_errors` on purpose — with the well-formed file, a malformed row should fail loudly, not vanish silently.

`loan_product.csv` and `credit_pull.csv` each have one genuinely truncated row near the end of the file (found by direct inspection, not guessed) with no well-formed backup available, so `ignore_errors=true` quarantines just that one row instead of crashing the whole load.

In [7]:
with open('applicant.csv') as f:
    lines = f.readlines()

expected_fields = len(lines[0].strip().split(','))
clean_lines = [lines[0]]
for line in lines[1:]:
    fields = line.rstrip('\n').split(',')
    if len(fields) < expected_fields:
        fields += [''] * (expected_fields - len(fields))
        print(f"Padded a short row to {expected_fields} fields: {fields[0]}")
    clean_lines.append(','.join(fields) + '\n')

with open('applicant_clean.csv', 'w') as f:
    f.writelines(clean_lines)

con.execute("""
    CREATE OR REPLACE TABLE _staging_applicants AS
    SELECT * FROM read_csv('applicant_clean.csv', delim=',')
""")

con.execute("SELECT COUNT(*) FROM _staging_applicants").df()

Padded a short row to 12 fields: APP-00060


,count_star()
0,60


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# The 'SOL' row (line 20) was intentionally left incomplete by the team as
# one of the assignment's required controlled quality problems (Part 6).
# Remove it explicitly, rather than relying on ignore_errors to silently skip it.
with open('loan_product.csv') as f:
    lines = f.readlines()

expected_fields = len(lines[0].strip().split(','))
clean_lines = [lines[0]] + [
    line for line in lines[1:] if len(line.strip().split(',')) == expected_fields
]
removed = len(lines) - len(clean_lines)
print(f"Removed {removed} intentionally-incomplete row(s) from loan_product.csv")

with open('loan_product_clean.csv', 'w') as f:
    f.writelines(clean_lines)

con.execute("""
    CREATE OR REPLACE TABLE _staging_loan_products AS
    SELECT * FROM read_csv('loan_product_clean.csv', delim=',')
""")

con.execute("SELECT COUNT(*) FROM _staging_loan_products").df()


Removed 1 intentionally-incomplete row(s) from loan_product.csv


,count_star()
0,18


In [10]:
con.execute("""
    CREATE OR REPLACE TABLE _staging_credit_pull AS
    SELECT * FROM read_csv('credit_pull.csv', delim=',', ignore_errors=true)
""")

con.execute("SELECT COUNT(*) FROM _staging_credit_pull").df()

,count_star()
0,75


## Step 2: Build constrained tables (real primary/foreign keys)

Unlike the staging tables above, these have actual `PRIMARY KEY` and `REFERENCES` constraints — DuckDB will reject bad inserts instead of silently accepting them.

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE loan_products (
        product_code VARCHAR PRIMARY KEY,
        product_name VARCHAR,
        product_family VARCHAR,
        eligibility_criteria VARCHAR,
        minimum_credit_tier VARCHAR,
        minimum_fico BIGINT,
        base_rate_pct DOUBLE,
        maximum_amount_usd BIGINT
    )
""")
con.execute("INSERT INTO loan_products SELECT * FROM _staging_loan_products")

con.execute("SELECT * FROM loan_products").df()

,product_code,product_name,product_family,eligibility_criteria,minimum_credit_tier,minimum_fico,base_rate_pct,maximum_amount_usd
0,PL-STD-01,Standard Personal Loan,personal,US resident 18+; DTI<=45%; min 12mo employment...,T3,620,11.99,40000
1,PL-PRM-02,Prime Personal Loan,personal,Tier T1-T2 only; DTI<=40%; min 24mo employment...,T2,680,7.49,75000
2,PL-SUB-03,Subprime Personal Loan,personal,Tier T4 allowed with cosigner or collateral; D...,T4,580,24.99,15000
3,AUTO-NEW-01,New Auto Loan,auto,New vehicle <=1yr; LTV<=115%; min tier T3; min...,T3,620,6.49,85000
4,AUTO-USED-02,Used Auto Loan,auto,Vehicle <=10yr; LTV<=125%; min tier T3,T3,620,8.29,60000
5,AUTO-REFI-03,Auto Refinance,auto,Existing auto loan seasoned >=6mo; min tier T2,T2,680,6.99,70000
6,HELOC-01,Home Equity Line of Credit,mortgage,Owner-occupied primary; CLTV<=85%; min tier T2...,T2,680,8.75,250000
7,MORT-CONV-01,Conventional Mortgage,mortgage,Conforming limit; LTV<=97%; DTI<=43% ATR/QM; m...,T2,680,6.85,766550
8,MORT-FHA-02,FHA Mortgage,mortgage,FHA-insured; LTV<=96.5%; min FICO 580; DTI<=50...,T3,580,6.55,498257
9,MORT-JMB-03,Jumbo Mortgage,mortgage,Above conforming; LTV<=80%; min tier T1; reser...,T1,740,7.20,3000000


In [12]:
con.execute("""
    CREATE OR REPLACE TABLE applicants (
        applicant_id VARCHAR PRIMARY KEY,
        annual_income_usd BIGINT,
        employment_duration_months BIGINT,
        credit_risk_tier VARCHAR,
        fico_score BIGINT,
        region_state VARCHAR,
        requested_amount_usd BIGINT,
        requested_product_code VARCHAR REFERENCES loan_products(product_code),
        dti_pct DOUBLE,
        military_scra_flag BIGINT,
        bankruptcy_last_7y_flag BIGINT,
        preliminary_decision VARCHAR
    )
""")

**Two corrections applied on insert:**
1. `HELOAN-01` → `HELOC-01` (confirmed typo — no `HELOAN-01` product exists, `HELOC-01` does).
2. `SOLAR-01` rows excluded — that product's definition is the truncated row in `loan_product.csv`, so there's nothing valid to reference yet.

**Watch the WHERE clause carefully:** it uses `IS DISTINCT FROM`, not `!=`. With `!=`, any row where `requested_product_code` is `NULL` (like `APP-00060`) would get silently dropped too — `NULL != 'SOLAR-01'` evaluates to `NULL` in SQL, not `TRUE`, so a plain `WHERE ... != ...` filters it out along with the real target. `IS DISTINCT FROM` treats `NULL` as a real, comparable value instead.

In [13]:
con.execute("""
    INSERT INTO applicants
    SELECT
        applicant_id, annual_income_usd, employment_duration_months, credit_risk_tier,
        fico_score, region_state, requested_amount_usd,
        CASE WHEN requested_product_code = 'HELOAN-01' THEN 'HELOC-01'
             ELSE requested_product_code END,
        dti_pct, military_scra_flag, bankruptcy_last_7y_flag, preliminary_decision
    FROM _staging_applicants
    WHERE requested_product_code IS DISTINCT FROM 'SOLAR-01'
""")

con.execute("SELECT COUNT(*) FROM applicants").df()

,count_star()
0,59


In [14]:
con.execute("""
    CREATE OR REPLACE TABLE credit_pull (
        event_id VARCHAR PRIMARY KEY,
        applicant_id VARCHAR REFERENCES applicants(applicant_id),
        event_date DATE,
        prior_tier VARCHAR,
        new_tier VARCHAR,
        prior_fico BIGINT,
        new_fico BIGINT,
        tier_changed_flag BIGINT,
        bureau_source VARCHAR,
        event_status VARCHAR,
        event_reason VARCHAR
    )
""")
con.execute("INSERT INTO credit_pull SELECT * FROM _staging_credit_pull")

con.execute("SELECT COUNT(*) FROM credit_pull").df()

,count_star()
0,75


## Step 3: Quality reports (the "curated structured outputs")

Each of these is a validation query, run and inspected one at a time.

In [15]:
# Business-rule violation: requested more than the product's maximum
amount_exceeds_max = con.execute("""
    SELECT a.applicant_id, a.requested_product_code, a.requested_amount_usd,
           lp.maximum_amount_usd
    FROM applicants a
    JOIN loan_products lp
        ON a.requested_product_code = lp.product_code
    WHERE a.requested_amount_usd > lp.maximum_amount_usd
""").df()

amount_exceeds_max

,applicant_id,requested_product_code,requested_amount_usd,maximum_amount_usd
0,APP-00036,PL-PRM-02,95000,75000
1,APP-00038,CC-STD-02,11000,10000


In [16]:
# Rows with an unusually high number of NULL fields (e.g. APP-00060)
nullable_columns = [
    "annual_income_usd", "employment_duration_months", "credit_risk_tier",
    "fico_score", "region_state", "requested_amount_usd",
    "requested_product_code", "dti_pct", "military_scra_flag",
    "bankruptcy_last_7y_flag", "preliminary_decision",
]
null_count_expr = " + ".join(f"CASE WHEN {c} IS NULL THEN 1 ELSE 0 END" for c in nullable_columns)

incomplete_records = con.execute(f"""
    SELECT applicant_id, ({null_count_expr}) AS null_field_count
    FROM applicants
    WHERE ({null_count_expr}) >= 3
    ORDER BY null_field_count DESC
""").df()

incomplete_records

,applicant_id,null_field_count
0,APP-00060,10


In [17]:
# Same applicant, more than one credit event on the same date
duplicate_credit_events = con.execute("""
    SELECT applicant_id, event_date, COUNT(*) AS event_count
    FROM credit_pull
    GROUP BY applicant_id, event_date
    HAVING COUNT(*) > 1
""").df()

duplicate_credit_events

,applicant_id,event_date,event_count


In [18]:
# Everything quarantined/excluded anywhere in the pipeline, with why
quarantine_summary = con.execute("""
    SELECT 'loan_product.csv' AS source_file, 'line 20 ("SOL")' AS location,
           'intentionally incomplete row (Part 6 controlled quality problem)' AS reason
    UNION ALL
    SELECT 'credit_pull.csv', 'line 77 ("EVT-000076,APP-00")',
           'truncated row, cut off mid-value'
    UNION ALL
    SELECT 'applicant.csv (via applicants)', 'APP-00032 (SOLAR-01 reference)',
           'excluded: requested_product_code has no matching product'
""").df()

quarantine_summary

,source_file,location,reason
0,loan_product.csv,"line 20 (""SOL"")",intentionally incomplete row (Part 6 controlle...
1,credit_pull.csv,"line 77 (""EVT-000076,APP-00"")","truncated row, cut off mid-value"
2,applicant.csv (via applicants),APP-00032 (SOLAR-01 reference),excluded: requested_product_code has no matchi...


## Step 4: Export reports and download

In [19]:
import os
os.makedirs("reports", exist_ok=True)

amount_exceeds_max.to_csv("reports/amount_exceeds_max.csv", index=False)
incomplete_records.to_csv("reports/incomplete_records.csv", index=False)
duplicate_credit_events.to_csv("reports/duplicate_credit_events.csv", index=False)
quarantine_summary.to_csv("reports/quarantine_summary.csv", index=False)

print("Reports written to reports/")

Reports written to reports/


In [20]:
from google.colab import files

for f in ["amount_exceeds_max", "incomplete_records", "duplicate_credit_events", "quarantine_summary"]:
    files.download(f"reports/{f}.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>